# 06b — Fine-tuning on ADNI v2 (122 Subjects)
### NeuroReach | Mohina Rustamova
**Goal:** Fine-tune 3-class ViT on labeled ADNI v2 data.
- 122 subjects × 20 slices = 2440 samples
- 5-fold cross validation
- Compare: ViT fine-tuned vs ViT+CORAL+fine-tuned
- Expected accuracy: 80%+

In [ ]:
# ============================================================
# Cell 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import os
print('Google Drive mounted.')

In [ ]:
# ============================================================
# Cell 2 — Setup Kaggle API
# ============================================================
from google.colab import files
print('Upload your kaggle.json:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('/content/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle API configured.')

In [ ]:
# ============================================================
# Cell 3 — Install Dependencies + Imports
# ============================================================
!pip install timm pydicom -q

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from torchvision import transforms
from collections import defaultdict
from pathlib import Path
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report, confusion_matrix
import pydicom
import pandas as pd
import timm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# Cell 4 — Download ADNI v2
# ============================================================
!kaggle datasets download -d mohinarustamova/neuroreach-adni-v2 -p /content/adni_v2 --unzip
print('ADNI v2 downloaded.')

# Check structure
count = 0
for root, dirs, files_list in os.walk('/content/adni_v2'):
    for f in files_list[:1]:
        print(os.path.join(root, f))
        count += 1
    if count >= 5:
        break

In [ ]:
# ============================================================
# Cell 5 — Load ADNI v2 Labels + Build Labeled Dataset
# ============================================================
# Update paths after checking Cell 4 output
ADNI_ROOT = '/content/adni_v2/neuroreach_adni_v2/ADNI'
CSV_PATH  = '/content/adni_v2/neuroreach_adni_v2_6_25_2026.csv'

CLASS_NAMES_3 = ['CN (Normal)', 'MCI', 'AD (Alzheimers)']
GROUP_MAP     = {'CN': 0, 'MCI': 1, 'AD': 2}

# Load labels
df = pd.read_csv(CSV_PATH)
subject_labels   = df[['Subject', 'Group']].drop_duplicates('Subject')
subject_to_label = {
    row['Subject']: GROUP_MAP[row['Group']]
    for _, row in subject_labels.iterrows()
}
print(f'Subjects with labels: {len(subject_to_label)}')
print(f'Distribution: {pd.Series(subject_to_label.values()).value_counts().to_dict()}')

# Collect DICOM files
adni_files  = []
subject_ids = []
for dcm_path in Path(ADNI_ROOT).rglob('*.dcm'):
    subject_id = dcm_path.parts[-5]
    adni_files.append(str(dcm_path))
    subject_ids.append(subject_id)

subject_to_files = defaultdict(list)
for path, subj in zip(adni_files, subject_ids):
    subject_to_files[subj].append(path)
for subj in subject_to_files:
    subject_to_files[subj].sort()

print(f'Total DICOM files : {len(adni_files)}')
print(f'Unique subjects   : {len(subject_to_files)}')

# Transforms
adni_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class ADNILabeledDataset(Dataset):
    def __init__(self, subject_to_files, subject_to_label,
                 slices_per_subject=20, transform=None):
        self.transform = transform
        self.samples   = []
        for subj, files in subject_to_files.items():
            if subj not in subject_to_label:
                continue
            label  = subject_to_label[subj]
            n      = len(files)
            start  = int(0.20 * n)
            end    = int(0.80 * n)
            usable = files[start:end]
            indices = np.linspace(0, len(usable) - 1, slices_per_subject, dtype=int)
            for i in indices:
                self.samples.append((usable[i], label))
        print(f'Total labeled samples: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        dcm = pydicom.dcmread(path)
        img = dcm.pixel_array.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = np.stack([img, img, img], axis=0)
        img = torch.tensor(img, dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, label

# Build dataset with 20 slices per subject
adni_full = ADNILabeledDataset(
    subject_to_files, subject_to_label,
    slices_per_subject=20, transform=adni_transforms
)
print(f'Dataset ready: {len(adni_full)} samples')

In [ ]:
# ============================================================
# Cell 6 — Model Builder Functions
# ============================================================
# Two model types:
#   A) ViT fine-tuned — loads vit_3class_oasis.pth
#   B) ViT+CORAL fine-tuned — loads vit_3class_coral_adapted.pth

VIT_PATH   = '/content/drive/MyDrive/NeuroReach/vit_3class_oasis.pth'
CORAL_PATH = '/content/drive/MyDrive/NeuroReach/vit_3class_coral_adapted.pth'

def build_vit(weights_path, unfreeze_blocks=2):
    vit = timm.create_model('vit_base_patch16_224', pretrained=False)
    vit.head = nn.Linear(vit.head.in_features, 3)
    vit.load_state_dict(torch.load(weights_path, map_location=device), strict=False)
    # Freeze all
    for param in vit.parameters():
        param.requires_grad = False
    # Unfreeze last N blocks + head
    for block in vit.blocks[-unfreeze_blocks:]:
        for param in block.parameters():
            param.requires_grad = True
    for param in vit.head.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in vit.parameters() if p.requires_grad)
    return vit.to(device), trainable

# Test both load correctly
vit_test, params = build_vit(VIT_PATH)
print(f'ViT fine-tuned     — trainable params: {params:,}')
vit_test, params = build_vit(CORAL_PATH)
print(f'ViT+CORAL fine-tuned — trainable params: {params:,}')

In [ ]:
# ============================================================
# Cell 7 — 5-Fold CV: ViT Fine-tuned on ADNI v2
# ============================================================
def run_kfold(weights_path, model_name, n_splits=5, epochs=20, lr=1e-4):
    kf      = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    indices = list(range(len(adni_full)))
    fold_accuracies = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(indices)):
        print(f'\n--- Fold {fold+1}/{n_splits} ---')

        train_loader = DataLoader(adni_full, batch_size=16,
                                  sampler=SubsetRandomSampler(train_idx))
        test_loader  = DataLoader(adni_full, batch_size=16,
                                  sampler=SubsetRandomSampler(test_idx))

        model, _ = build_vit(weights_path)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()

        for epoch in range(epochs):
            model.train()
            correct, total = 0, 0
            for imgs, lbls in train_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss    = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()
                preds    = outputs.argmax(dim=1)
                correct += (preds == lbls).sum().item()
                total   += lbls.size(0)
            if (epoch + 1) % 5 == 0:
                print(f'  Epoch {epoch+1}/{epochs} Train Acc: {correct/total*100:.2f}%')

        # Evaluate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs = imgs.to(device)
                outputs = model(imgs)
                preds   = outputs.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(lbls.numpy())

        acc = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
        fold_accuracies.append(acc)
        print(f'Fold {fold+1} Test Accuracy: {acc:.2f}%')

    print(f'\n=== {model_name} — 5-Fold CV Results ===')
    print(f'Per fold  : {[f"{a:.2f}%" for a in fold_accuracies]}')
    print(f'Mean      : {np.mean(fold_accuracies):.2f}%')
    print(f'Std       : {np.std(fold_accuracies):.2f}%')
    return fold_accuracies

# Run ViT fine-tuned
print('Running ViT fine-tuned on ADNI v2...')
accs_vit = run_kfold(VIT_PATH, 'ViT fine-tuned')

In [ ]:
# ============================================================
# Cell 8 — 5-Fold CV: ViT+CORAL Fine-tuned on ADNI v2
# ============================================================
print('Running ViT+CORAL fine-tuned on ADNI v2...')
accs_coral = run_kfold(CORAL_PATH, 'ViT+CORAL fine-tuned')

In [ ]:
# ============================================================
# Cell 9 — Final Comparison + Save Results
# ============================================================
import json

print('=' * 60)
print('NEUROREACH MODULE 1 — COMPLETE RESULTS')
print('=' * 60)
print(f'{"Model":<35} {"Dataset":<10} {"Accuracy"}')
print('-' * 60)
print(f'{"ResNet18":<35} {"OASIS":<10} 99.89%')
print(f'{"ViT 3-class":<35} {"OASIS":<10} 99.85%')
print(f'{"ResNet18":<35} {"ADNI":<10} 30.37%')
print(f'{"ViT + CORAL (4-class)":<35} {"ADNI":<10} 35.56%')
print(f'{"ViT fine-tuned (27 subj)":<35} {"ADNI":<10} 70.56% ± 4.77%')
print(f'{"ViT fine-tuned (122 subj)":<35} {"ADNI":<10} {np.mean(accs_vit):.2f}% ± {np.std(accs_vit):.2f}%')
print(f'{"ViT+CORAL fine-tuned (122 subj)":<35} {"ADNI":<10} {np.mean(accs_coral):.2f}% ± {np.std(accs_coral):.2f}%')
print('=' * 60)

# Save results
results = {
    'resnet18_oasis'                  : 99.89,
    'vit_3class_oasis'                : 99.85,
    'resnet18_adni_zeroshot'          : 30.37,
    'vit_coral_4class_adni'           : 35.56,
    'vit_finetuned_27subj_mean'       : 70.56,
    'vit_finetuned_27subj_std'        : 4.77,
    'vit_finetuned_122subj_mean'      : round(np.mean(accs_vit), 2),
    'vit_finetuned_122subj_std'       : round(np.std(accs_vit), 2),
    'vit_coral_finetuned_122subj_mean': round(np.mean(accs_coral), 2),
    'vit_coral_finetuned_122subj_std' : round(np.std(accs_coral), 2),
    'vit_finetuned_122subj_folds'     : accs_vit,
    'vit_coral_finetuned_122subj_folds': accs_coral,
}

with open('/content/drive/MyDrive/NeuroReach/module1_results_final.json', 'w') as f:
    json.dump(results, f, indent=4)
print('\nFinal results saved to Google Drive!')